# Highway Lane Analytics - Colab Activation

Jalankan sel instalasi, lalu **pilih salah satu** sel opsi:
1. **ZIP Google Drive** - 12 video @ 5 menit
2. **Live stream** - rekam HLS lalu proses

Hasil batch otomatis dikirim ke pipeline produksi lewat **Cloudflare Tunnel**
(`https://lane-api.syaifudin-fam.my.id`), bukan ngrok, untuk mensimulasikan feed
near-real-time. Token diambil dari Colab Secrets (`PIPELINE_WEBHOOK_TOKEN`).


In [ ]:
import os
import subprocess

REPO = "https://github.com/ibrahim119920/highway-lane-analytics.git"
PROJECT = "highway-lane-analytics"
if not os.path.isdir(PROJECT):
    subprocess.run(["git", "clone", REPO, PROJECT], check=True)
%cd highway-lane-analytics
# Ambil perubahan terbaru (tuning + sender pipeline).
subprocess.run(["git", "pull", "--ff-only"], check=False)
!pip install -q -e ".[colab]"

## Opsi 1 — ZIP 12 video dari Google Drive

Isi `drive_url` dengan link publik zip. `base_timestamp` opsional; jika diisi, 12 klip
dianggap satu timeline berurutan (durasi tiap klip ditambahkan otomatis).

In [ ]:
from lane_analytics.colab import activate

activate(
    source="drive_zip",
    drive_url="https://drive.google.com/file/d/XXXXXXXX/view?usp=sharing",
    base_timestamp="2026-11-07T23:00:00",   # opsional; kosongkan untuk infer dari nama file
    # Token dibaca dari Colab Secrets: PIPELINE_WEBHOOK_TOKEN
    # Opsional: PIPELINE_BASE_URL bila hostname pipeline berubah.
    speed=5,          # 1 = real-time, 5 = 5x lebih cepat
    window_seconds=5, # ukuran micro-batch dalam detik event-time
)


## Opsi 2 — Live stream HLS

Stream direkam dulu selama `record_seconds`, lalu diproses dan di-replay.
`base_timestamp` diambil dari waktu mulai rekaman.

In [ ]:
from lane_analytics.colab import activate, DEFAULT_STREAM_URL

activate(
    source="live",
    stream_url=DEFAULT_STREAM_URL,
    record_seconds=300,
    # Token dibaca dari Colab Secrets: PIPELINE_WEBHOOK_TOKEN
    speed=1,
)


## Catatan

- Receiver webhook, Kafka, pipeline, dan dashboard berada di proyek terpisah.
- URL default adalah pipeline produksi melalui Cloudflare Tunnel. Ganti lewat
  `webhook_url=` atau secret `PIPELINE_BASE_URL` bila perlu.
- Token **wajib** ada di Colab Secrets sebagai `PIPELINE_WEBHOOK_TOKEN`; jangan
  menulisnya di notebook. Bila belum ada, `activate()` berhenti dengan pesan jelas.
- Untuk 12 klip dengan nama file yang tidak memuat tanggal, isi `base_timestamps={...}`
  agar timestamp event akurat.
- `speed=5` mempercepat replay; `timestamp` event tidak berubah.
- Sender menolak `timestamp` ber-offset (`Z`/`+07:00`); event harus WIB naive.
- Snapshot + klip 8 detik per event otomatis ditulis ke `output/evidence/<run_id>/`.
- Proses tanpa replay: `activate(..., replay=False)`.
